# DeepWalk

## Learning Objectives

1. **Explain** the analogy between random walks on graphs and sentences in text
2. **Describe** the Skip-Gram model and how it learns node embeddings
3. **State** the DeepWalk objective and the role of negative sampling
4. **Identify** what graph structure is captured by DeepWalk embeddings
5. **Implement** random walk generation and Skip-Gram training from scratch


## Problem Statement

### Why Embed Nodes?

Many graph ML tasks (node classification, link prediction, clustering) require feature vectors for nodes. Traditional approaches use hand-crafted features (degree, clustering coefficient, etc.) that don't generalise.

**Node embedding:** Learn a mapping $\text{ENC}: V \to \mathbb{R}^d$ such that nodes with similar graph-neighbourhood structure have similar embeddings.

### DeepWalk's Insight

Text corpora and graphs share a statistical property: **short random walks** on graphs follow a power-law frequency distribution over node sequences, similar to word frequencies in natural language.

Therefore, apply **Word2Vec (Skip-Gram)** to node sequences generated by random walks.

**Encoder-Decoder framework:**
- Encoder: $z_v = \text{embed}(v)$ — lookup row in embedding matrix
- Decoder: $\text{DEC}(z_u, z_v) = \sigma(z_u^\top z_v)$ — dot product similarity
- Similarity: $\Pr[u \in \mathcal{N}_R(v)]$ — probability of $u$ appearing in random walks from $v$


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <defs><marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#999"/></marker></defs>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">DeepWalk — Random Walks + Skip-Gram on Graphs</text>

  <!-- Graph -->
  <rect x="20" y="36" width="250" height="200" rx="5" fill="#e8f0fb" stroke="#4e79a7"/>
  <text x="145" y="56" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">1. Graph G</text>
  <circle cx="80"  cy="110" r="14" fill="#4e79a7"/><text x="80"  cy="114" text-anchor="middle" fill="white" font-size="10">v₁</text>
  <circle cx="145" cy="80"  r="14" fill="#4e79a7"/><text x="145" cy="84"  text-anchor="middle" fill="white" font-size="10">v₂</text>
  <circle cx="210" cy="110" r="14" fill="#4e79a7"/><text x="210" cy="114" text-anchor="middle" fill="white" font-size="10">v₃</text>
  <circle cx="80"  cy="175" r="14" fill="#59a14f"/><text x="80"  cy="179" text-anchor="middle" fill="white" font-size="10">v₄</text>
  <circle cx="145" cy="200" r="14" fill="#59a14f"/><text x="145" cy="204" text-anchor="middle" fill="white" font-size="10">v₅</text>
  <circle cx="210" cy="175" r="14" fill="#59a14f"/><text x="210" cy="179" text-anchor="middle" fill="white" font-size="10">v₆</text>
  <line x1="80" y1="110" x2="131" y2="84"  stroke="#ccc" stroke-width="1.5"/>
  <line x1="80" y1="110" x2="196" y2="110" stroke="#ccc" stroke-width="1.5"/>
  <line x1="145" y1="80" x2="196" y2="110" stroke="#ccc" stroke-width="1.5"/>
  <line x1="80" y1="124" x2="80" y2="161"  stroke="#ccc" stroke-width="1.5"/>
  <line x1="80" y1="175" x2="131" y2="200" stroke="#ccc" stroke-width="1.5"/>
  <line x1="210" y1="124" x2="210" y2="161" stroke="#ccc" stroke-width="1.5"/>
  <line x1="145" y1="200" x2="196" y2="175" stroke="#ccc" stroke-width="1.5"/>
  <!-- walk path highlighted -->
  <line x1="145" y1="80"  x2="84"  y2="107" stroke="#e15759" stroke-width="2.5"/>
  <line x1="84"  y1="107" x2="94"  y2="163" stroke="#e15759" stroke-width="2.5"/>
  <line x1="94"  y1="163" x2="139" y2="196" stroke="#e15759" stroke-width="2.5"/>
  <line x1="139" y1="196" x2="196" y2="175" stroke="#e15759" stroke-width="2.5"/>
  <text x="145" y="228" text-anchor="middle" fill="#e15759" font-size="10">Walk: v₂→v₁→v₄→v₅→v₆</text>

  <!-- Arrow to Skip-Gram -->
  <line x1="272" y1="136" x2="316" y2="136" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Skip-Gram window -->
  <rect x="320" y="36" width="280" height="200" rx="5" fill="#fff4e0" stroke="#f28e2b"/>
  <text x="460" y="56" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">2. Skip-Gram (word2vec) on walk</text>
  <text x="460" y="76" text-anchor="middle" fill="#333" font-size="11">Walk = "sentence";  Nodes = "words"</text>
  <!-- Token sequence -->
  <rect x="330" y="90" width="40" height="22" rx="2" fill="#e15759"/><text x="350" y="105" text-anchor="middle" fill="white" font-size="10">v₂</text>
  <rect x="378" y="90" width="40" height="22" rx="2" fill="#e15759"/><text x="398" y="105" text-anchor="middle" fill="white" font-size="10">v₁</text>
  <rect x="426" y="90" width="40" height="22" rx="2" fill="#4e79a7" stroke="#333" stroke-width="2"/><text x="446" y="105" text-anchor="middle" fill="white" font-size="10">v₄</text>
  <rect x="474" y="90" width="40" height="22" rx="2" fill="#e15759"/><text x="494" y="105" text-anchor="middle" fill="white" font-size="10">v₅</text>
  <rect x="522" y="90" width="40" height="22" rx="2" fill="#e15759"/><text x="542" y="105" text-anchor="middle" fill="white" font-size="10">v₆</text>
  <!-- window brackets -->
  <rect x="378" y="86" width="178" height="32" rx="3" fill="none" stroke="#f28e2b" stroke-width="2" stroke-dasharray="4,2"/>
  <text x="467" y="136" text-anchor="middle" fill="#f28e2b" font-size="10">Context window w=2 around v₄</text>
  <!-- objective -->
  <text x="460" y="162" text-anchor="middle" fill="#333" font-size="11">Maximize:</text>
  <text x="460" y="180" text-anchor="middle" fill="#333" font-size="11">Σ log P(context | center node)</text>
  <text x="460" y="198" text-anchor="middle" fill="#555" font-size="10">= learn d-dim embeddings z_v for each node</text>
  <text x="460" y="218" text-anchor="middle" fill="#555" font-size="10">P(u|v) = exp(zᵤᵀzᵥ) / Σₖ exp(zₖᵀzᵥ)</text>

  <!-- Arrow to embeddings -->
  <line x1="602" y1="136" x2="646" y2="136" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Embedding space -->
  <rect x="650" y="36" width="148" height="200" rx="5" fill="#e8f8e8" stroke="#59a14f"/>
  <text x="724" y="56" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">3. Embeddings</text>
  <circle cx="700" cy="100" r="10" fill="#4e79a7"/>
  <text x="714" y="98" fill="#4e79a7" font-size="10">v₁</text>
  <circle cx="730" cy="90"  r="10" fill="#4e79a7"/>
  <text x="744" y="88" fill="#4e79a7" font-size="10">v₂</text>
  <circle cx="715" cy="120" r="10" fill="#4e79a7"/>
  <text x="729" y="118" fill="#4e79a7" font-size="10">v₃</text>
  <circle cx="690" cy="165" r="10" fill="#59a14f"/>
  <text x="704" y="163" fill="#59a14f" font-size="10">v₄</text>
  <circle cx="720" cy="175" r="10" fill="#59a14f"/>
  <text x="734" y="173" fill="#59a14f" font-size="10">v₅</text>
  <circle cx="740" cy="158" r="10" fill="#59a14f"/>
  <text x="754" y="156" fill="#59a14f" font-size="10">v₆</text>
  <text x="724" y="222" text-anchor="middle" fill="#555" font-size="10">Communities cluster together</text>

  <!-- Params -->
  <rect x="20" y="248" width="780" height="104" rx="5" fill="#f5f5f5" stroke="#ccc"/>
  <text x="410" y="268" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Hyperparameters &amp; Training</text>
  <text x="200" y="290" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Walk generation</text>
  <text x="200" y="308" text-anchor="middle" fill="#555" font-size="10">γ = walks per node (e.g. 80)</text>
  <text x="200" y="322" text-anchor="middle" fill="#555" font-size="10">t = walk length (e.g. 40)</text>
  <text x="200" y="336" text-anchor="middle" fill="#555" font-size="10">Uniform random transitions</text>
  <text x="410" y="290" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Skip-Gram</text>
  <text x="410" y="308" text-anchor="middle" fill="#555" font-size="10">d = embedding dim (e.g. 128)</text>
  <text x="410" y="322" text-anchor="middle" fill="#555" font-size="10">w = window size (e.g. 10)</text>
  <text x="410" y="336" text-anchor="middle" fill="#555" font-size="10">Hierarchical softmax or neg. sampling</text>
  <text x="650" y="290" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Applications</text>
  <text x="650" y="308" text-anchor="middle" fill="#555" font-size="10">Node classification</text>
  <text x="650" y="322" text-anchor="middle" fill="#555" font-size="10">Link prediction</text>
  <text x="650" y="336" text-anchor="middle" fill="#555" font-size="10">Community detection</text>
  <text x="650" y="350" text-anchor="middle" fill="#555" font-size="10">Graph visualization</text>
</svg>
'''
display(SVG(svg))


## Derivation

### Random Walk Objective

For each node $v$, generate $\gamma$ random walks of length $t$. The Skip-Gram objective maximises:
$$\max_{z} \sum_{v \in V} \sum_{u \in \mathcal{N}_R^w(v)} \log P(u \mid z_v)$$
where $\mathcal{N}_R^w(v)$ = nodes within window $w$ of $v$ across all walks, and:
$$P(u \mid z_v) = \frac{\exp(z_u^\top z_v)}{\sum_{k \in V} \exp(z_k^\top z_v)}$$

The softmax over all $|V|$ nodes is expensive — use **hierarchical softmax** (original paper) or **negative sampling** (faster, more common).

### Negative Sampling

For each positive pair $(v, u)$: sample $K$ negative nodes $\{n_k\} \sim P_n$ and optimise:
$$\log \sigma(z_u^\top z_v) + \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n} \left[\log \sigma(-z_{n_k}^\top z_v)\right]$$

### What DeepWalk Captures

Nodes that frequently co-occur in short random walks → close in embedding space. This captures **community structure** (homophily): nodes in the same dense community tend to appear in the same walks.

**Limitation:** Transductive — cannot embed unseen nodes without retraining. Also does not use node features. Node2Vec extends with biased walks; GraphSAGE extends to inductive settings.

### Complexity

- Walk generation: $O(\gamma |V| t)$
- Skip-Gram training: $O(|\text{walks}| \cdot w \cdot (d + K))$
- Total: $O(\gamma |V| t (w d + Kw))$


## Algorithm Steps

1. **Input:** Graph $G$, dimension $d$, walks per node $\gamma$, walk length $t$, window $w$
2. **Initialise** embedding matrix $Z \in \mathbb{R}^{|V| \times d}$ randomly
3. **For each epoch:**
   - Shuffle nodes; for each node $v_0$: generate random walk $[v_0, v_1, \ldots, v_t]$
   - For each centre $v_i$ and context $v_j$ within window $w$: SGD update via Skip-Gram with negative sampling
4. **Output** embedding matrix $Z$


In [ ]:
import numpy as np
import random
from collections import defaultdict


def random_walk(graph, start, length):
    """Uniform random walk of fixed length starting at `start`."""
    walk = [start]
    for _ in range(length - 1):
        curr = walk[-1]
        neighbors = list(graph[curr].keys())
        if not neighbors:
            break
        walk.append(random.choice(neighbors))
    return walk


def generate_walks(graph, n_walks, walk_length, seed=42):
    """Generate corpus of random walks over all nodes."""
    random.seed(seed)
    nodes = list(graph.keys())
    walks = []
    for _ in range(n_walks):
        random.shuffle(nodes)
        for v in nodes:
            walks.append(random_walk(graph, v, walk_length))
    return walks


def skipgram_sgd(walks, n_nodes, dim=32, window=3, n_iter=3, lr=0.025, neg_samples=5, seed=42):
    """
    Skip-Gram with Negative Sampling (simplified).

    For each centre node and context within window:
    - Maximise dot product for true context
    - Minimise for k random negative samples
    """
    rng = np.random.default_rng(seed)
    # Two embedding matrices: centre and context
    W = rng.uniform(-0.5/dim, 0.5/dim, (n_nodes, dim))   # centre embeddings
    C = rng.zeros((n_nodes, dim))                         # context embeddings

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    node_freq = np.ones(n_nodes)   # uniform negative sampling for demo
    neg_prob  = node_freq / node_freq.sum()

    for epoch in range(n_iter):
        total_loss = 0.0
        for walk in walks:
            for i, centre in enumerate(walk):
                start = max(0, i - window)
                end   = min(len(walk), i + window + 1)
                for j in range(start, end):
                    if j == i:
                        continue
                    context = walk[j]
                    # Positive pair
                    score = sigmoid(W[centre] @ C[context])
                    grad  = (1 - score) * lr
                    C[context] += grad * W[centre]
                    W[centre]  += grad * C[context]
                    total_loss -= np.log(score + 1e-9)
                    # Negative samples
                    negs = rng.choice(n_nodes, size=neg_samples, p=neg_prob)
                    for neg in negs:
                        if neg == context:
                            continue
                        score_neg = sigmoid(W[centre] @ C[neg])
                        grad_neg  = -score_neg * lr
                        C[neg]    += grad_neg * W[centre]
                        W[centre] += grad_neg * C[neg]
                        total_loss -= np.log(1 - score_neg + 1e-9)
        print(f"Epoch {epoch+1}/{n_iter}, loss: {total_loss:.1f}")

    return W   # final node embeddings


def deepwalk(graph, dim=32, n_walks=10, walk_length=20, window=3, seed=42):
    """Full DeepWalk pipeline."""
    n_nodes = len(graph)
    walks = generate_walks(graph, n_walks, walk_length, seed=seed)
    embeddings = skipgram_sgd(walks, n_nodes, dim=dim, window=window, seed=seed)
    return embeddings


# ── Demo ──────────────────────────────────────────────────────────────────────
import random as rnd; rnd.seed(0)

# Build two cliques (communities) with a bridge
def build_graph(n, edges):
    g = defaultdict(dict)
    for u, v in edges:
        g[u][v] = 1.0
        g[v][u] = 1.0
    return dict(g)

clique1 = [(i, j) for i in range(0, 6)  for j in range(i+1, 6)]
clique2 = [(i, j) for i in range(6, 12) for j in range(i+1, 12)]
bridge  = [(5, 6)]
graph = build_graph(12, clique1 + clique2 + bridge)

emb = deepwalk(graph, dim=8, n_walks=20, walk_length=15, window=3, seed=42)

# Verify: nodes in same clique should have similar embeddings
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

sim_same = np.mean([cosine_sim(emb[i], emb[j]) for i in range(0,6) for j in range(i+1,6)])
sim_diff = np.mean([cosine_sim(emb[i], emb[j]) for i in range(0,6) for j in range(6,12)])
print(f"\nAvg cosine sim within clique: {sim_same:.4f}")
print(f"Avg cosine sim across cliques: {sim_diff:.4f}")
print("DeepWalk succeeds if within-clique > across-clique similarity")
